In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

In [2]:
riders_raw = pd.read_csv(RAW_DATA_DIR/"riders.csv")
agents_raw = pd.read_csv(RAW_DATA_DIR/"agents.csv")
trips_raw = pd.read_csv(RAW_DATA_DIR/"trips.csv")

support_tickets_raw = pd.read_csv(RAW_DATA_DIR/"support_tickets.csv")
support_events_raw = pd.read_csv(RAW_DATA_DIR/"support_event_tickets.csv")

financial_adjustments_raw = pd.read_csv(RAW_DATA_DIR/"financial_adjustments.csv")
customer_feedback_raw = pd.read_csv(RAW_DATA_DIR/"customer_feedback.csv")
experiment_assignments_raw = pd.read_csv(RAW_DATA_DIR/"experiment_assignments.csv")

In [3]:
# Look at column and row counts
datasets = {
    "riders": riders_raw,
    "agents": agents_raw,
    "trips": trips_raw,
    "support_tickets": support_tickets_raw,
    "support_events": support_events_raw,
    "financial_adjustments": financial_adjustments_raw,
    "customer_feedback": customer_feedback_raw,
    "experiment_assignments": experiment_assignments_raw
}

for name, df in datasets.items():
    print(
        f"{name:<25} "
        f"rows={len(df):>8,} "
        f"columns={len(df.columns):>3}"
    )

riders                    rows=  25,000 columns=  4
agents                    rows=     120 columns=  6
trips                     rows= 120,000 columns= 11
support_tickets           rows=  14,697 columns= 11
support_events            rows= 113,021 columns=  7
financial_adjustments     rows=   6,106 columns=  8
customer_feedback         rows=   6,760 columns=  6
experiment_assignments    rows=   1,748 columns=  6


In [4]:
# What are the data types
for name, df in datasets.items():
    print("\n")
    print(name.upper())
    print("=" * 60)
    print(df.dtypes)



RIDERS
rider_id            str
signup_date         str
home_city           str
customer_segment    str
dtype: object


AGENTS
agent_id             str
support_team         str
tenure_months      int64
cost_per_hour    float64
location             str
active_flag         bool
dtype: object


TRIPS
trip_id             str
rider_id            str
city                str
ride_type           str
requested_at        str
started_at          str
completed_at        str
cancelled_at        str
trip_status         str
fare_amount     float64
distance_km     float64
dtype: object


SUPPORT_TICKETS
ticket_id           str
trip_id             str
rider_id            str
parent_ticket_id    str
opened_at           str
resolved_at         str
issue_type          str
channel             str
priority            str
status              str
initial_agent_id    str
dtype: object


SUPPORT_EVENTS
event_id              str
ticket_id             str
event_time            str
event_type            str
agent

In [5]:
# validate aside  financial_adjustment, all other dataframes don't have duplicate rows
primary_keys = {
    "riders": "rider_id",
    "agents": "agent_id",
    "trips": "trip_id",
    "support_tickets": "ticket_id",
    "support_events": "event_id",
    "financial_adjustments": "adjustment_id",
    "customer_feedback": "feedback_id",
    "experiment_assignments": "assignment_id"
}

for name, primary_key in primary_keys.items():
    df = datasets[name]
    duplicate_count = df[primary_key].duplicated().sum()

    print(
        f"{name:<25}"
        f"{primary_key:<20}"
        f"duplicates={duplicate_count}"
    )

riders                   rider_id            duplicates=0
agents                   agent_id            duplicates=0
trips                    trip_id             duplicates=0
support_tickets          ticket_id           duplicates=0
support_events           event_id            duplicates=0
financial_adjustments    adjustment_id       duplicates=25
customer_feedback        feedback_id         duplicates=0
experiment_assignments   assignment_id       duplicates=0


In [6]:
# Look for missing values

for name, df in datasets.items():
    null_counts = df.isna().sum()
    null_counts = null_counts[null_counts > 0]

    print("\n")
    print(name.upper())
    print("=" * 60)

    if null_counts.empty:
        print("No missing values")
    else:
        print(null_counts)



RIDERS
No missing values


AGENTS
No missing values


TRIPS
started_at        6329
completed_at     10642
cancelled_at    109358
dtype: int64


SUPPORT_TICKETS
trip_id                30
parent_ticket_id    12576
dtype: int64


SUPPORT_EVENTS
agent_id    30423
dtype: int64


FINANCIAL_ADJUSTMENTS
No missing values


CUSTOMER_FEEDBACK
No missing values


EXPERIMENT_ASSIGNMENTS
No missing values


In [7]:
# Are there trips with missing Ids
missing_ticket_trip_ids = support_tickets_raw["trip_id"].isna().sum()
print(f"Support tickets with missing trip_id: {missing_ticket_trip_ids}")

Support tickets with missing trip_id: 30


In [8]:
# Does any ticket have a non-existent trip id?

valid_trip_ids = set(trips_raw["trip_id"])

orphan_ticket_trips = support_tickets_raw[
    support_tickets_raw["trip_id"].notna()
    &
    ~support_tickets_raw["trip_id"].isin(valid_trip_ids)
    ]
print(f"Tickets referencing nonexistent trips: {len(orphan_ticket_trips)}")

Tickets referencing nonexistent trips: 0


In [9]:
# Are there support events that are not affiliated with tickets?
valid_ticket_ids = set(support_tickets_raw["ticket_id"])

orphan_support_events = (
    support_events_raw[~support_events_raw["ticket_id"].isin(valid_ticket_ids)]
)
print(f"Orphan support events: {len(orphan_support_events)}")

Orphan support events: 10


In [10]:
# Let us have a feel of what the orphan support events look like?
orphan_support_events[["event_id", "ticket_id", "event_type"]].head(10)

,event_id,ticket_id,event_type
113011,EVT0113012,TK_ORPHAN_001,escalated
113012,EVT0113013,TK_ORPHAN_002,agent_reply
113013,EVT0113014,TK_ORPHAN_003,agent_reply
113014,EVT0113015,TK_ORPHAN_004,agent_reply
113015,EVT0113016,TK_ORPHAN_005,assigned
113016,EVT0113017,TK_ORPHAN_006,customer_reply
113017,EVT0113018,TK_ORPHAN_007,agent_reply
113018,EVT0113019,TK_ORPHAN_008,resolved
113019,EVT0113020,TK_ORPHAN_009,customer_reply
113020,EVT0113021,TK_ORPHAN_010,customer_reply


In [11]:
support_tickets_check = support_tickets_raw.copy()

support_tickets_check["opened_at"] = pd.to_datetime(support_tickets_check["opened_at"], format="mixed", errors="coerce")
support_tickets_check["resolved_at"] = pd.to_datetime(support_tickets_check["resolved_at"], format="mixed", errors="coerce")

invalid_ticket_timestamps = (
    support_tickets_check[support_tickets_check["resolved_at"] < support_tickets_check["opened_at"]]
)
print(f"Tickets resolved before opening: {len(invalid_ticket_timestamps)}")

Tickets resolved before opening: 10


In [12]:
invalid_csat = customer_feedback_raw[~customer_feedback_raw["csat_score"].between(1, 5)]
print(f"Invalid CSAT records: {len(invalid_csat)}")

Invalid CSAT records: 10


In [13]:
invalid_csat["csat_score"].value_counts()

csat_score
6    10
Name: count, dtype: int64

In [14]:
# Create a dataframe of the validation issues

validation_summary = pd.DataFrame({
    "check": [
        "Duplicate financial adjustment IDs",
        "Support tickets missing trip_id",
        "Support tickets referencing nonexistent trips",
        "Orphan support events",
        "Tickets resolved before opened",
        "Invalid CSAT scores"
    ],
    "issue_count": [
        financial_adjustments_raw["adjustment_id"].duplicated().sum(),
        support_tickets_raw["trip_id"].isna().sum(),
        len(orphan_ticket_trips),
        len(orphan_support_events),
        len(invalid_ticket_timestamps),
        len(invalid_csat)
    ],
})

validation_summary

,check,issue_count
0,Duplicate financial adjustment IDs,25
1,Support tickets missing trip_id,30
2,Support tickets referencing nonexistent trips,0
3,Orphan support events,10
4,Tickets resolved before opened,10
5,Invalid CSAT scores,10


In [17]:
# Verify that a repeat ticket has a parent ticket id

repeat_tickets = support_tickets_raw[support_tickets_raw["parent_ticket_id"].notna()]
valid_ticket_ids = set(support_events_raw["ticket_id"])
invalid_parent_tickets = repeat_tickets[~repeat_tickets["parent_ticket_id"].isin(valid_ticket_ids)]

print(f"Repeat tickets: {len(repeat_tickets)}")
print(f"Repeat tickets with invalid parent: {len(invalid_parent_tickets)}")

Repeat tickets: 2121
Repeat tickets with invalid parent: 0


In [19]:
# Let us verify that a ticket does not have a parent id pointing to itself
self_referencing_tickets = support_tickets_raw[support_tickets_raw["ticket_id"] == support_tickets_raw["parent_ticket_id"]]
print(f"Self-referencing tickets: {len(self_referencing_tickets)}")

Self-referencing tickets: 0


In [22]:
# Verify that no trip marked as completed had null in the following columns: started_at, completed_at or cancelled_at

trips_check = trips_raw.copy()
timestamp_columns = ["requested_at", "started_at", "completed_at", "cancelled_at"]

for column in timestamp_columns:
    trips_check[column] = pd.to_datetime(trips_check[column], format="mixed", errors="coerce")

completed_trips = trips_check[trips_check["trip_status"] == "completed"]
bad_completed_trips = completed_trips[
    completed_trips["started_at"].isna() | completed_trips["completed_at"].isna() | completed_trips["cancelled_at"].notna()
]

print(f"Invalid completed trips: {len(bad_completed_trips)}")

Invalid completed trips: 0


In [24]:
# Any cancelled trip with a completion date/time?

cancelled_trips = trips_check[trips_check["trip_status"] == "cancelled"]
bad_cancelled_trips = cancelled_trips[
    cancelled_trips["cancelled_at"].isna() | cancelled_trips["completed_at"].notna()
]

print(f"Invalid cancelled trips: {len(bad_cancelled_trips)}")

Invalid cancelled trips: 0


In [26]:
# Any customer_feedback which does not point to an existing ticket?

invalid_feedback_tickets = customer_feedback_raw[
    ~customer_feedback_raw["ticket_id"].isin(valid_ticket_ids)
]

print(f"Feedback referencing nonexistent tickets: {len(invalid_feedback_tickets)}")

Feedback referencing nonexistent tickets: 0


In [28]:
# Do all rider ids mentioned in a feedback exist?
valid_rider_ids = set(riders_raw["rider_id"])

invalid_feedback_riders = customer_feedback_raw[
    ~customer_feedback_raw["rider_id"].isin(valid_rider_ids)
]

print(f"Feedback referencing nonexistent riders: {len(invalid_feedback_riders)}")

Feedback referencing nonexistent riders: 0


In [29]:
# Each experiment assignment should point to a ticket that exists
invalid_experiment_tickets = experiment_assignments_raw[
    ~experiment_assignments_raw["ticket_id"].isin(valid_ticket_ids)
]

print(f"Experiment assignments with invalid ticket: {len(invalid_experiment_tickets)}")

Experiment assignments with invalid ticket: 0


In [30]:
experiment_assignments_raw["experiment_group"].value_counts()

experiment_group
Control      882
Treatment    866
Name: count, dtype: int64

In [31]:
# Verify there is no assignment before 1st April 2026
experiment_check = experiment_assignments_raw.copy()

experiment_check["assigned_at"] = pd.to_datetime(experiment_check["assigned_at"], format="mixed", errors="coerce")
pre_launch_assignments = experiment_check[experiment_check["assigned_at"] < pd.Timestamp("2026-04-01")]

print(f"Pre-launch experiment assignments: {len(pre_launch_assignments)}")

Pre-launch experiment assignments: 0


In [32]:
# Is there a refund/appeasement without a ticket

valid_ticket_ids = set(support_events_raw["ticket_id"])

invalid_adjustment_tickets = (
    financial_adjustments_raw[~financial_adjustments_raw["ticket_id"].isin(valid_ticket_ids)]
)
print(f"Financial adjustments with invalid ticket: {len(invalid_adjustment_tickets)}")

Financial adjustments with invalid ticket: 0


In [35]:
# Each refund/appeasement should be associated with a trip that exists

valid_trip_ids = set(trips_raw["trip_id"])

invalid_adjustment_trips = (
    financial_adjustments_raw[~financial_adjustments_raw["trip_id"].isin(valid_trip_ids)]
)
print(f"Financial adjustments with invalid trip: {len(invalid_adjustment_trips)}")
print(financial_adjustments_raw["adjustment_type"].value_counts())

Financial adjustments with invalid trip: 0
adjustment_type
refund                4144
appeasement_credit    1962
Name: count, dtype: int64


In [37]:
# Do refund/appeasements have negative or zero amounts?

invalid_adjustment_amounts = financial_adjustments_raw[financial_adjustments_raw["amount"] <= 0]
print(f"Zero/negative financial adjustments: {len(invalid_adjustment_amounts)}")

Zero/negative financial adjustments: 0


In [38]:
# Are refunds greater than fare for the trip?

refund_check = (
    financial_adjustments_raw[financial_adjustments_raw["adjustment_type"] == "refund"]
    .merge(trips_raw[["trip_id", "fare_amount"]], on="trip_id", how="left")
)
refunds_above_fare = refund_check[refund_check["amount"] > refund_check["fare_amount"]]

print(f"Refunds greater than trip fare: {len(refunds_above_fare)}")

Refunds greater than trip fare: 0


In [40]:
# Verify that customer response events will not have an agent id

support_events_raw[support_events_raw["agent_id"].isna()]["event_type"].value_counts()

event_type
customer_reply    30423
Name: count, dtype: int64

In [42]:
# Support events which have no agent ids

valid_agent_ids = set(agents_raw["agent_id"])

invalid_event_agents = support_events_raw[
    support_events_raw["agent_id"].notna() &
    ~support_events_raw["agent_id"].isin(valid_agent_ids)
]

print(f"Support events with invalid agent: {len(invalid_event_agents)}")

Support events with invalid agent: 0


In [43]:
# The agent id on a support ticket exists

invalid_initial_agents = (
    support_tickets_raw[~support_tickets_raw["initial_agent_id"].isin(valid_agent_ids)]
)

print(f"Support tickets with invalid initial agent: {len(invalid_initial_agents)}")

Support tickets with invalid initial agent: 0


In [47]:
# Validate that all experiment assignment tickets were for AVV ride type

experiment_eligiblity_check = (
    experiment_assignments_raw
    .merge(support_tickets_raw[["ticket_id", "trip_id", "issue_type"]], on="ticket_id", how="left")
    .merge(trips_raw[["trip_id", "ride_type"]], on="trip_id", how="left")
)

unverifiable_experiment_tickets = (experiment_eligiblity_check[experiment_eligiblity_check["ride_type"].isna()])

true_non_av_experiment_tickets = experiment_eligiblity_check[
    experiment_eligiblity_check["ride_type"].notna() &
    (experiment_eligiblity_check["ride_type"] != "AV")]

print(f"Experiment assignment with missing trip relationship: {len(unverifiable_experiment_tickets)}")
print(f"Confirmed non-AV experiment assignments: {len(true_non_av_experiment_tickets)}")

Experiment assignment with missing trip relationship: 6
Confirmed non-AV experiment assignments: 0


In [45]:
eligible_issues = {"pickup_issue", "vehicle_access_issue", "trip_status_issue", "cancellation_issue"}

invalid_experiment_issues = (
    experiment_eligiblity_check[~experiment_eligiblity_check["issue_type"].isin(eligible_issues)]
)

print(f"Experiment assignment with ineligible issue: {len(invalid_experiment_issues)}")

Experiment assignment with ineligible issue: 0


In [46]:
experiment_eligiblity_check[experiment_eligiblity_check["ride_type"] != "AV"][["ticket_id", "trip_id", "ride_type", "issue_type"]]

,ticket_id,trip_id,ride_type,issue_type
68,TK0000450,NaN,NaN,trip_status_issue
233,TK0001645,NaN,NaN,vehicle_access_issue
771,TK0005444,NaN,NaN,pickup_issue
987,TK0006941,NaN,NaN,cancellation_issue
1310,TK0009351,NaN,NaN,vehicle_access_issue
1445,TK0010405,NaN,NaN,pickup_issue
